<a href="https://colab.research.google.com/github/macsrc/mac-handson-ml3/blob/handson-ml-241025/ch15_explore.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import sys
assert sys.version_info >= (3,7)

In [2]:
from packaging import version
import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")

In [3]:
import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

In [4]:
from pathlib import Path

IMAGE_PATH = Path()/ "images"/ "rnn"
IMAGE_PATH.mkdir(parents=True, exist_ok=True)

def save_fig(fig_id, tight_layout=True, fig_extension="png", resolution=300):
  path= IMAGE_PATH/ f"{fig_id}.{fig_extension}"
  if tight_layout:
    plt.tight_layout
  plt.savefig(path, fig_extension, dpi=resolution)


In [5]:
if not tf.config.list_physical_devices('GPU'):
  print("No GPU was detected. Neural nets can be very slow without a GPU.")
  if "google.colab" in sys.modules:
    print("Go to Runtime > Change runtime and select a GPU hardware accelerator")
  if "kaggle_secrets" in sys.modules:
    print("Go to Settings > Accelerator and select GPU.")

No GPU was detected. Neural nets can be very slow without a GPU.
Go to Runtime > Change runtime and select a GPU hardware accelerator


# Basic RNNs

In [10]:
import tarfile
from pathlib import Path

filepath = tf.keras.utils.get_file(
    "ridership.tgz",
    "https://github.com/ageron/data/raw/main/ridership.tgz",
    cache_dir="./datasets", # Changed cache_dir to ./datasets
    # extracted=True # Removed this argument as it causes a TypeError
)

# Manually extract the tar.gz file
# filepath will be something like './datasets/ridership.tgz'
with tarfile.open(filepath, "r:gz") as tar:
    tar.extractall(path=Path(filepath).parent) # Extracts to the same directory as the tgz file, i.e., './datasets'

# ridership_path will be './datasets/ridership' after extraction
ridership_path = Path(filepath).with_name("ridership")

/tmp/ipython-input-2906737596.py:14: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=Path(filepath).parent) # Extracts to the same directory as the tgz file, i.e., './datasets'


In [11]:
import pandas as pd
from pathlib import Path

path = ridership_path / "CTA_-_Ridership_-_Daily_Boarding_Totals.csv"
df = pd.read_csv(path, parse_dates=["service_date"])
df.columns = ["date", "day_type", "bus", "rail", "total"]  # shorter names
df = df.sort_values("date").set_index("date")
df = df.drop("total", axis=1)  # no need for total, it's just bus + rail
df = df.drop_duplicates()  # remove duplicated months (2011-10 and 2014-07)

In [12]:
import pandas as pd
from pathlib import Path

path = ridership_path / "CTA_-_Ridership_-_Daily_Boarding_Totals.csv"
df = pd.read_csv(path, parse_dates=["service_date"])
df.columns = ["date", "day_type", "bus", "rail", "total"]  # shorter names
df = df.sort_values("date").set_index("date")
df = df.drop("total", axis=1)  # no need for total, it's just bus + rail
df = df.drop_duplicates()  # remove duplicated months (2011-10 and 2014-07)

In [13]:
import pandas as pd
from pathlib import Path

# path - "CTA_-_Ridership_-_Daily_Boarding_Totals.csv"
path = ridership_path/"CTA_-_Ridership_-_Daily_Boarding_Totals.csv"
df = pd.read_csv(path, parse_dates=["service_date"])
df.columns = ["date", "day_type", "bus", "rail", "total"]
df = df.sort_values("date").set_index("date")
df = df.drop("total", axis=1)
df = df.drop_duplicates()
